In [1]:
from __future__ import annotations

import os
from datetime import UTC, datetime, timedelta
from getpass import getpass
from typing import Any

import httpx
import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
vehicles = [
    {
        "year": 2022,
        "make": "Toyota",
        "model": "Corolla",
        "segment": "compact sedan",
    },
    {
        "year": 2022,
        "make": "Honda",
        "model": "Civic",
        "segment": "compact sedan",
    },
    {
        "year": 2022,
        "make": "Ford",
        "model": "F-150",
        "segment": "pickup truck",
    },
    {
        "year": 2022,
        "make": "Tesla",
        "model": "Model 3",
        "segment": "electric sedan",
    },
    {
        "year": 2022,
        "make": "Toyota",
        "model": "RAV4",
        "segment": "compact SUV",
    },
]

In [5]:
def get_json(
    url: str,
    *,
    params: dict[str, Any] | None = None,
) -> dict[str, Any]:
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json",
    }

    with httpx.Client(
        timeout=30.0,
        follow_redirects=True,
        headers=headers,
    ) as http_client:
        response = http_client.get(url, params=params)
        response.raise_for_status()
        return response.json()

In [6]:
def fetch_fuel_economy_options(
    *,
    year: int,
    make: str,
    model: str,
) -> list[dict[str, Any]]:
    url = "https://www.fueleconomy.gov/ws/rest/vehicle/menu/options"

    data = get_json(
        url,
        params={
            "year": year,
            "make": make,
            "model": model,
        },
    )

    menu_item = data.get("menuItem", [])

    if isinstance(menu_item, dict):
        return [menu_item]

    if isinstance(menu_item, list):
        return menu_item

    return []

In [7]:
def fetch_fuel_economy_vehicle(vehicle_id: str) -> dict[str, Any]:
    url = f"https://www.fueleconomy.gov/ws/rest/vehicle/{vehicle_id}"

    return get_json(url)

In [8]:
def fetch_nhtsa_recalls(
    *,
    year: int,
    make: str,
    model: str,
) -> list[dict[str, Any]]:
    url = "https://api.nhtsa.gov/recalls/recallsByVehicle"

    data = get_json(
        url,
        params={
            "modelYear": year,
            "make": make,
            "model": model,
        },
    )

    results = data.get("results") or data.get("Results") or []

    if isinstance(results, dict):
        return [results]

    return results

In [9]:
def to_float(value: Any, default: float = 0.0) -> float:
    if value is None:
        return default

    if value == "":
        return default

    try:
        return float(value)
    except ValueError:
        return default


def to_int(value: Any, default: int = 0) -> int:
    if value is None:
        return default

    if value == "":
        return default

    try:
        return int(float(value))
    except ValueError:
        return default


def get_first_text(value: Any, default: str = "unknown") -> str:
    if value is None:
        return default

    if value == "":
        return default

    return str(value)

In [10]:
def normalize_fuel_economy_data(
    *,
    vehicle_config: dict[str, Any],
    fuel_data: dict[str, Any],
) -> dict[str, Any]:
    city_mpg = to_float(fuel_data.get("city08"))
    highway_mpg = to_float(fuel_data.get("highway08"))
    combined_mpg = to_float(fuel_data.get("comb08"))

    co2 = to_float(fuel_data.get("co2TailpipeGpm"))
    annual_fuel_cost = to_float(fuel_data.get("fuelCost08"))

    fuel_type = get_first_text(fuel_data.get("fuelType"))
    vehicle_class = get_first_text(fuel_data.get("VClass"))
    drive = get_first_text(fuel_data.get("drive"))
    transmission = get_first_text(fuel_data.get("trany"))
    cylinders = to_int(fuel_data.get("cylinders"))
    displacement = to_float(fuel_data.get("displ"))

    if combined_mpg >= 40:
        efficiency_label = "very efficient"
    elif combined_mpg >= 30:
        efficiency_label = "efficient"
    elif combined_mpg >= 22:
        efficiency_label = "moderate efficiency"
    elif combined_mpg > 0:
        efficiency_label = "low efficiency"
    else:
        efficiency_label = "unknown efficiency"

    if co2 == 0:
        emissions_label = "zero tailpipe emissions or unknown CO2"
    elif co2 <= 250:
        emissions_label = "lower CO2"
    elif co2 <= 400:
        emissions_label = "moderate CO2"
    else:
        emissions_label = "higher CO2"

    fuel_summary = (
        f"{vehicle_config['year']} {vehicle_config['make']} {vehicle_config['model']} "
        f"is classified as {vehicle_class}. Fuel type: {fuel_type}. "
        f"City MPG: {city_mpg}, highway MPG: {highway_mpg}, combined MPG: {combined_mpg}. "
        f"Annual fuel cost estimate: {annual_fuel_cost}. "
        f"CO2 tailpipe grams per mile: {co2}. "
        f"Drive: {drive}. Transmission: {transmission}. "
        f"Engine: {displacement}L, cylinders: {cylinders}. "
        f"Efficiency label: {efficiency_label}. Emissions label: {emissions_label}."
    )

    return {
        "fuel_vehicle_id": get_first_text(fuel_data.get("id")),
        "vehicle_class": vehicle_class,
        "fuel_type": fuel_type,
        "city_mpg": city_mpg,
        "highway_mpg": highway_mpg,
        "combined_mpg": combined_mpg,
        "annual_fuel_cost": annual_fuel_cost,
        "co2_tailpipe_gpm": co2,
        "drive": drive,
        "transmission": transmission,
        "cylinders": cylinders,
        "displacement_l": displacement,
        "efficiency_label": efficiency_label,
        "emissions_label": emissions_label,
        "fuel_summary": fuel_summary,
    }

In [11]:
def normalize_recall_data(recalls: list[dict[str, Any]]) -> dict[str, Any]:
    recall_count = len(recalls)

    components = []
    campaigns = []
    recall_text_parts = []

    for recall in recalls:
        campaign = get_first_text(
            recall.get("NHTSACampaignNumber")
            or recall.get("CampaignNumber")
            or recall.get("nhtsaCampaignNumber")
        )

        component = get_first_text(
            recall.get("Component")
            or recall.get("component")
        )

        summary = get_first_text(
            recall.get("Summary")
            or recall.get("summary")
        )

        consequence = get_first_text(
            recall.get("Consequence")
            or recall.get("consequence")
        )

        remedy = get_first_text(
            recall.get("Remedy")
            or recall.get("remedy")
        )

        campaigns.append(campaign)
        components.append(component)

        recall_text_parts.append(
            f"Campaign {campaign}. Component: {component}. "
            f"Summary: {summary}. Consequence: {consequence}. Remedy: {remedy}."
        )

    unique_components = sorted(set(item for item in components if item != "unknown"))

    if recall_count == 0:
        safety_label = "no recalls found"
    elif recall_count <= 2:
        safety_label = "low recall count"
    elif recall_count <= 5:
        safety_label = "moderate recall count"
    else:
        safety_label = "high recall count"

    recall_summary = (
        f"Recall count: {recall_count}. "
        f"Safety label: {safety_label}. "
        f"Main components: {', '.join(unique_components) if unique_components else 'none found'}. "
        + " ".join(recall_text_parts[:5])
    )

    return {
        "recall_count": recall_count,
        "recall_components": unique_components,
        "recall_campaigns": campaigns,
        "safety_label": safety_label,
        "recall_summary": recall_summary,
    }

In [12]:
def build_vehicle_record(vehicle_config: dict[str, Any]) -> dict[str, Any]:
    print(
        "Fetching:",
        vehicle_config["year"],
        vehicle_config["make"],
        vehicle_config["model"],
    )

    options = fetch_fuel_economy_options(
        year=vehicle_config["year"],
        make=vehicle_config["make"],
        model=vehicle_config["model"],
    )

    if not options:
        raise RuntimeError(f"No FuelEconomy options found for: {vehicle_config}")

    selected_option = options[0]
    fuel_vehicle_id = str(selected_option["value"])

    fuel_data = fetch_fuel_economy_vehicle(fuel_vehicle_id)

    recalls = fetch_nhtsa_recalls(
        year=vehicle_config["year"],
        make=vehicle_config["make"],
        model=vehicle_config["model"],
    )

    fuel_normalized = normalize_fuel_economy_data(
        vehicle_config=vehicle_config,
        fuel_data=fuel_data,
    )

    recalls_normalized = normalize_recall_data(recalls)

    combined_summary = (
        f"Vehicle intelligence profile for {vehicle_config['year']} "
        f"{vehicle_config['make']} {vehicle_config['model']}. "
        f"Segment: {vehicle_config['segment']}. "
        f"{fuel_normalized['fuel_summary']} "
        f"{recalls_normalized['recall_summary']}"
    )

    return {
        "year": vehicle_config["year"],
        "make": vehicle_config["make"],
        "model": vehicle_config["model"],
        "segment": vehicle_config["segment"],
        **fuel_normalized,
        **recalls_normalized,
        "combined_summary": combined_summary,
    }

In [13]:
vehicle_records = []

for vehicle_config in vehicles:
    try:
        record = build_vehicle_record(vehicle_config)
        vehicle_records.append(record)
    except Exception as error:
        print("Failed:", vehicle_config)
        print(error)

print("Vehicle records:", len(vehicle_records))

Fetching: 2022 Toyota Corolla
Failed: {'year': 2022, 'make': 'Toyota', 'model': 'Corolla', 'segment': 'compact sedan'}
Client error '400 Bad Request' for url 'https://api.nhtsa.gov/recalls/recallsByVehicle?modelYear=2022&make=Toyota&model=Corolla'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400
Fetching: 2022 Honda Civic
Failed: {'year': 2022, 'make': 'Honda', 'model': 'Civic', 'segment': 'compact sedan'}
'NoneType' object has no attribute 'get'
Fetching: 2022 Ford F-150
Failed: {'year': 2022, 'make': 'Ford', 'model': 'F-150', 'segment': 'pickup truck'}
'NoneType' object has no attribute 'get'
Fetching: 2022 Tesla Model 3
Failed: {'year': 2022, 'make': 'Tesla', 'model': 'Model 3', 'segment': 'electric sedan'}
'NoneType' object has no attribute 'get'
Fetching: 2022 Toyota RAV4
Vehicle records: 1


In [14]:
for record in vehicle_records:
    print(record["year"], record["make"], record["model"])
    print("Fuel:", record["fuel_type"])
    print("Combined MPG:", record["combined_mpg"])
    print("Recall count:", record["recall_count"])
    print("Safety label:", record["safety_label"])
    print("Summary:", record["combined_summary"][:1000])
    print("-" * 100)

2022 Toyota RAV4
Fuel: Regular
Combined MPG: 30.0
Recall count: 1
Safety label: low recall count
Summary: Vehicle intelligence profile for 2022 Toyota RAV4. Segment: compact SUV. 2022 Toyota RAV4 is classified as Small Sport Utility Vehicle 2WD. Fuel type: Regular. City MPG: 27.0, highway MPG: 35.0, combined MPG: 30.0. Annual fuel cost estimate: 2250.0. CO2 tailpipe grams per mile: 296.0. Drive: Front-Wheel Drive. Transmission: Automatic (S8). Engine: 2.5L, cylinders: 4. Efficiency label: efficient. Emissions label: moderate CO2. Recall count: 1. Safety label: low recall count. Main components: AIR BAGS:SENSOR:OCCUPANT CLASSIFICATION:FRONT PASSENGER. Campaign 22V519000. Component: AIR BAGS:SENSOR:OCCUPANT CLASSIFICATION:FRONT PASSENGER. Summary: Toyota Motor Engineering & Manufacturing (Toyota) is recalling certain 2022 Rav4, Rav4 Hybrid, and Rav4 Prime vehicles.  The front passenger seat may have been assembled with interference between internal parts that may cause the Occupant Class

In [15]:
COLLECTION_NAME = "VehicleIntelligenceProfile"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

vehicle_profiles = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(name="year", data_type=wvc.config.DataType.INT),
        wvc.config.Property(name="make", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="model", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="segment", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="fuel_vehicle_id", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="vehicle_class", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="fuel_type", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="city_mpg", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="highway_mpg", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="combined_mpg", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="annual_fuel_cost", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="co2_tailpipe_gpm", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="drive", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="transmission", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="cylinders", data_type=wvc.config.DataType.INT),
        wvc.config.Property(name="displacement_l", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="efficiency_label", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="emissions_label", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="recall_count", data_type=wvc.config.DataType.INT),
        wvc.config.Property(name="recall_components", data_type=wvc.config.DataType.TEXT_ARRAY),
        wvc.config.Property(name="recall_campaigns", data_type=wvc.config.DataType.TEXT_ARRAY),
        wvc.config.Property(name="safety_label", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="fuel_summary", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="recall_summary", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="combined_summary", data_type=wvc.config.DataType.TEXT),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: VehicleIntelligenceProfile


In [16]:
if not vehicle_records:
    raise RuntimeError("No vehicle records to import.")

result = vehicle_profiles.data.insert_many(vehicle_records)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [17]:
response = vehicle_profiles.aggregate.over_all(total_count=True)

print("Total objects:", response.total_count)

Total objects: 1


In [18]:
response = vehicle_profiles.query.fetch_objects(
    limit=10,
    return_properties=[
        "year",
        "make",
        "model",
        "segment",
        "fuel_type",
        "combined_mpg",
        "annual_fuel_cost",
        "recall_count",
        "safety_label",
        "efficiency_label",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("Vehicle:", obj.properties["year"], obj.properties["make"], obj.properties["model"])
    print("Segment:", obj.properties["segment"])
    print("Fuel:", obj.properties["fuel_type"])
    print("Combined MPG:", obj.properties["combined_mpg"])
    print("Annual fuel cost:", obj.properties["annual_fuel_cost"])
    print("Recall count:", obj.properties["recall_count"])
    print("Safety:", obj.properties["safety_label"])
    print("Efficiency:", obj.properties["efficiency_label"])
    print("-" * 100)

UUID: d35376f7-da1e-4d0c-b61e-b749055bec4b
Vehicle: 2022 Toyota RAV4
Segment: compact SUV
Fuel: Regular
Combined MPG: 30.0
Annual fuel cost: 2250.0
Recall count: 1
Safety: low recall count
Efficiency: efficient
----------------------------------------------------------------------------------------------------


In [19]:
response = vehicle_profiles.query.near_text(
    query="efficient reliable car with low fuel cost and low recall count",
    limit=5,
    return_properties=[
        "year",
        "make",
        "model",
        "segment",
        "fuel_type",
        "combined_mpg",
        "annual_fuel_cost",
        "recall_count",
        "safety_label",
        "efficiency_label",
        "combined_summary",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Vehicle:", obj.properties["year"], obj.properties["make"], obj.properties["model"])
    print("Combined MPG:", obj.properties["combined_mpg"])
    print("Annual fuel cost:", obj.properties["annual_fuel_cost"])
    print("Recall count:", obj.properties["recall_count"])
    print("Safety:", obj.properties["safety_label"])
    print("Efficiency:", obj.properties["efficiency_label"])
    print("-" * 100)

Distance: 0.5585430860519409
Vehicle: 2022 Toyota RAV4
Combined MPG: 30.0
Annual fuel cost: 2250.0
Recall count: 1
Safety: low recall count
Efficiency: efficient
----------------------------------------------------------------------------------------------------


In [20]:
response = vehicle_profiles.query.fetch_objects(
    filters=Filter.by_property("combined_mpg").greater_or_equal(30),
    limit=10,
    return_properties=[
        "year",
        "make",
        "model",
        "fuel_type",
        "combined_mpg",
        "efficiency_label",
        "annual_fuel_cost",
    ],
)

for obj in response.objects:
    print("Vehicle:", obj.properties["year"], obj.properties["make"], obj.properties["model"])
    print("Fuel:", obj.properties["fuel_type"])
    print("Combined MPG:", obj.properties["combined_mpg"])
    print("Efficiency:", obj.properties["efficiency_label"])
    print("Annual fuel cost:", obj.properties["annual_fuel_cost"])
    print("-" * 100)

Vehicle: 2022 Toyota RAV4
Fuel: Regular
Combined MPG: 30.0
Efficiency: efficient
Annual fuel cost: 2250.0
----------------------------------------------------------------------------------------------------


In [21]:
response = vehicle_profiles.query.fetch_objects(
    filters=Filter.by_property("recall_count").greater_than(0),
    limit=10,
    return_properties=[
        "year",
        "make",
        "model",
        "recall_count",
        "recall_components",
        "safety_label",
        "recall_summary",
    ],
)

for obj in response.objects:
    print("Vehicle:", obj.properties["year"], obj.properties["make"], obj.properties["model"])
    print("Recall count:", obj.properties["recall_count"])
    print("Components:", obj.properties["recall_components"])
    print("Safety:", obj.properties["safety_label"])
    print("Recall summary:", obj.properties["recall_summary"][:800])
    print("-" * 100)

Vehicle: 2022 Toyota RAV4
Recall count: 1
Components: ['AIR BAGS:SENSOR:OCCUPANT CLASSIFICATION:FRONT PASSENGER']
Safety: low recall count
Recall summary: Recall count: 1. Safety label: low recall count. Main components: AIR BAGS:SENSOR:OCCUPANT CLASSIFICATION:FRONT PASSENGER. Campaign 22V519000. Component: AIR BAGS:SENSOR:OCCUPANT CLASSIFICATION:FRONT PASSENGER. Summary: Toyota Motor Engineering & Manufacturing (Toyota) is recalling certain 2022 Rav4, Rav4 Hybrid, and Rav4 Prime vehicles.  The front passenger seat may have been assembled with interference between internal parts that may cause the Occupant Classification System (OCS) sensor to incorrectly detect the occupant.  As such, these vehicles fail to comply with the requirements of Federal Motor Vehicle Safety Standard number 208, "Occupant Crash Protection.". Consequence: Incorrect detection of an occupant may result in improper air bag deployment during a crash, increasing the risk
--------------------------------------------

In [22]:
response = vehicle_profiles.query.hybrid(
    query="SUV truck fuel economy safety recalls",
    alpha=0.5,
    limit=5,
    return_properties=[
        "year",
        "make",
        "model",
        "segment",
        "fuel_type",
        "combined_mpg",
        "recall_count",
        "safety_label",
        "efficiency_label",
        "combined_summary",
    ],
    return_metadata=MetadataQuery(score=True),
)

for obj in response.objects:
    print("Hybrid score:", obj.metadata.score)
    print("Vehicle:", obj.properties["year"], obj.properties["make"], obj.properties["model"])
    print("Segment:", obj.properties["segment"])
    print("MPG:", obj.properties["combined_mpg"])
    print("Recalls:", obj.properties["recall_count"])
    print("Safety:", obj.properties["safety_label"])
    print("-" * 100)

Hybrid score: 1.0
Vehicle: 2022 Toyota RAV4
Segment: compact SUV
MPG: 30.0
Recalls: 1
Safety: low recall count
----------------------------------------------------------------------------------------------------


In [23]:
response = vehicle_profiles.generate.near_text(
    query="compare vehicles by fuel efficiency, emissions and recall safety",
    grouped_task=(
        "Compare the retrieved vehicles using only the provided records. "
        "Focus on fuel economy, annual fuel cost, CO2 label, recall count and safety label. "
        "Do not invent missing data. "
        "End with a short educational note that this is not purchasing advice."
    ),
    limit=5,
    return_properties=[
        "year",
        "make",
        "model",
        "segment",
        "fuel_type",
        "combined_mpg",
        "annual_fuel_cost",
        "co2_tailpipe_gpm",
        "efficiency_label",
        "emissions_label",
        "recall_count",
        "safety_label",
        "combined_summary",
    ],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print(
        "-",
        obj.properties["year"],
        obj.properties["make"],
        obj.properties["model"],
        "| MPG:",
        obj.properties["combined_mpg"],
        "| recalls:",
        obj.properties["recall_count"],
    )

Let's summarize the vehicle details based on the provided records focusing on fuel economy, annual fuel cost, CO2 label, recall count, and safety label. Here is the comparison for the 2022 Toyota RAV4:

- **Fuel Economy**:
  - **City MPG**: 27
  - **Highway MPG**: 35
  - **Combined MPG**: 30

- **Annual Fuel Cost**: $2,250

- **CO2 Label**: Moderate CO2 emissions (296 grams per mile)

- **Recall Count**: 1

- **Safety Label**: Low recall count

**Summary**: The 2022 Toyota RAV4 is categorized as a compact SUV, known for its decent fuel economy with a combined MPG of 30, making it relatively efficient for its class. The annual fuel cost is estimated at $2,250, and it has a low recall count, indicating better reliability and safety. The moderate CO2 emissions label suggests it is somewhat environmentally friendly, although there is still room for improvement.

**Educational Note**: This comparison is based solely on the provided records and does not constitute purchasing advice. Always c

In [24]:
def recommend_vehicle_profile(
    user_need: str,
    *,
    min_mpg: float | None = None,
    max_recall_count: int | None = None,
    segment: str | None = None,
    limit: int = 5,
) -> None:
    filters = None

    if min_mpg is not None:
        mpg_filter = Filter.by_property("combined_mpg").greater_or_equal(min_mpg)
        filters = mpg_filter if filters is None else filters & mpg_filter

    if max_recall_count is not None:
        recall_filter = Filter.by_property("recall_count").less_or_equal(max_recall_count)
        filters = recall_filter if filters is None else filters & recall_filter

    if segment is not None:
        segment_filter = Filter.by_property("segment").equal(segment)
        filters = segment_filter if filters is None else filters & segment_filter

    response = vehicle_profiles.generate.near_text(
        query=user_need,
        filters=filters,
        grouped_task=(
            "Act as an automotive data assistant. "
            "Use only the retrieved vehicle profiles. "
            "Recommend the best matching vehicle profile based on the user's need. "
            "Mention fuel economy, fuel cost, emissions label, recall count and safety label. "
            "Do not invent facts and do not provide financial or purchasing advice."
        ),
        limit=limit,
        return_properties=[
            "year",
            "make",
            "model",
            "segment",
            "fuel_type",
            "combined_mpg",
            "annual_fuel_cost",
            "co2_tailpipe_gpm",
            "efficiency_label",
            "emissions_label",
            "recall_count",
            "safety_label",
            "combined_summary",
        ],
    )

    print("USER NEED:", user_need)
    print("MIN MPG:", min_mpg)
    print("MAX RECALL COUNT:", max_recall_count)
    print("SEGMENT:", segment)
    print("-" * 100)

    print(response.generated)

    print("\nRecords used:")
    for obj in response.objects:
        print(
            "-",
            obj.properties["year"],
            obj.properties["make"],
            obj.properties["model"],
            "| MPG:",
            obj.properties["combined_mpg"],
            "| recalls:",
            obj.properties["recall_count"],
            "| segment:",
            obj.properties["segment"],
        )

In [25]:
recommend_vehicle_profile(
    "I want an efficient daily car with low fuel cost and low recall risk",
    min_mpg=30,
    max_recall_count=3,
)

USER NEED: I want an efficient daily car with low fuel cost and low recall risk
MIN MPG: 30
MAX RECALL COUNT: 3
SEGMENT: None
----------------------------------------------------------------------------------------------------
Based on the provided vehicle profile, the best matching vehicle is the **2022 Toyota RAV4**. Here are the relevant details:

- **Fuel Economy**: 
  - City MPG: 27
  - Highway MPG: 35
  - Combined MPG: 30
- **Annual Fuel Cost**: $2,250
- **Emissions Label**: Moderate CO2
- **Recall Count**: 1
- **Safety Label**: Low recall count

This compact SUV is known for its fuel efficiency and has a low recall count, indicating a generally good safety record. If you have specific preferences or requirements, please let me know for further recommendations!

Records used:
- 2022 Toyota RAV4 | MPG: 30.0 | recalls: 1 | segment: compact SUV


In [26]:
recommend_vehicle_profile(
    "I need a larger vehicle but want to understand fuel economy and safety recalls",
)

USER NEED: I need a larger vehicle but want to understand fuel economy and safety recalls
MIN MPG: None
MAX RECALL COUNT: None
SEGMENT: None
----------------------------------------------------------------------------------------------------
Based on your needs, I recommend the **2022 Toyota RAV4**. Here are the key details:

- **Fuel Economy**: 
  - City MPG: 27
  - Highway MPG: 35
  - Combined MPG: 30
- **Annual Fuel Cost**: $2,250
- **Emissions Label**: Moderate CO2
- **Recall Count**: 1
- **Safety Label**: Low recall count

The 2022 Toyota RAV4 is a compact SUV that offers a balance of efficiency and practicality, making it a great option for various driving needs. If you have any specific requirements or preferences, feel free to share!

Records used:
- 2022 Toyota RAV4 | MPG: 30.0 | recalls: 1 | segment: compact SUV


In [27]:
recommend_vehicle_profile(
    "I am interested in electric or low emissions vehicles",
)

USER NEED: I am interested in electric or low emissions vehicles
MIN MPG: None
MAX RECALL COUNT: None
SEGMENT: None
----------------------------------------------------------------------------------------------------
Based on the retrieved vehicle profile, I recommend the **2022 Toyota RAV4** for your needs. Here are the key specifications that match your criteria:

- **Fuel Economy**: 
  - City MPG: 27
  - Highway MPG: 35
  - Combined MPG: 30

- **Annual Fuel Cost**: Approximately $2,250

- **Emissions Label**: Moderate CO2

- **Recall Count**: 1 (considered a low recall count)

- **Safety Label**: Low recall count

This compact SUV offers a good balance of fuel efficiency and safety, making it suitable for various driving needs.

Records used:
- 2022 Toyota RAV4 | MPG: 30.0 | recalls: 1 | segment: compact SUV


In [28]:
client.close()